# CAR171 APT editor

Alejandro S. Borlaff, NASA ARC
Adapted from: Maxime Rizzo, NASA GSFC
Date: 5/10/26

In [ ]:
import xml.etree.ElementTree as ET
import copy
import pandas as pd
import astropy.units as u
from astropy.coordinates import SkyCoord

filename = 'CAR171.apt'
apt_file_dir = ''

CAR_dat = pd.read_csv("CAR171_apt_targets.csv")
CAR_dat
coords = SkyCoord(ra=CAR_dat['RA'].values*u.degree, dec=CAR_dat['DEC'].values*u.degree)

In [ ]:
CAR_dat

In [ ]:
coords[0].ra.to_string(unit=u.hour, sep=' ', precision=4, pad=True)

In [ ]:

import xml.etree.ElementTree as ET

def update_target_coordinates(xml_path, output_path, new_coords):
    """
    Updates RA/Dec coordinates for each FixedTarget in the XML.

    Parameters:
        xml_path (str): Path to the input XML file.
        output_path (str): Path to write the updated XML file.
        new_coords (list of tuples): List of (RA, Dec) strings.
            Example: [("13 47 3.30", "+49 01 40.49"), ...]
            Must be same length as number of FixedTarget entries.
    """

    tree = ET.parse(xml_path)
    root = tree.getroot()

    # XML uses namespaces; must extract them for searching.
    ns = {'ns': root.tag.split('}')[0].strip('{')}

    # Find all FixedTarget entries
    targets = root.findall('.//ns:FixedTarget', ns)

    if len(new_coords) != len(targets):
        raise ValueError(
            f"Provided {len(new_coords)} coordinates but XML contains {len(targets)} FixedTargets."
        )

    for (target, coord) in zip(targets, new_coords):
        ra = coord.ra.to_string(unit=u.hour, sep=' ', precision=4, pad=True)
        dec = coord.dec.to_string(unit=u.degree, sep=' ', precision=2, pad=True, alwayssign=True)
        # Find EquatorialCoordinates node
        eq = target.find('ns:EquatorialCoordinates', ns)
        if eq is not None:
            # Format must match XML's "Value" attribute: "RA Dec"
            eq.set("Value", f"{ra} {dec}")
            # print("Value", f"{ra} {dec}")
        else:
            print(f"Warning: FixedTarget missing EquatorialCoordinates element.")

    # Save modified XML
    tree.write(output_path, encoding="UTF-8", xml_declaration=True)
    print(f"Updated file written to: {output_path}")



def sync_passplan_numbers(xml_input, xml_output):
    """
    Updates each <PassPlan Number="X"> so that X matches its TargetSelection Fixed target ID.
    Example:
        <TargetSelection>Fixed: 13</TargetSelection>
        → PassPlan Number becomes "13".
    """

    tree = ET.parse(xml_input)
    root = tree.getroot()

    # Namespace used by Roman APT XML
    ns = {'ns': root.tag.split('}')[0].strip('{')}

    # Iterate through all PassPlan entries
    for passplan in root.findall('.//ns:PassPlan', ns):
        ts = passplan.find('ns:TargetSelection', ns)
        if ts is not None and "Fixed:" in ts.text:
            # Extract target number from "Fixed: N"
            target_num = ts.text.split("Fixed:")[1].strip()
            passplan.set("Number", target_num)

    # Save output
    tree.write(xml_output, encoding="UTF-8", xml_declaration=True)
    print(f"Updated XML saved to {xml_output}")



def sort_surveyplan_steps(xml_input, xml_output):
    """
    Sorts all <SurveyPlanStep> entries by their <PassPlan> value in increasing order.
    """

    tree = ET.parse(xml_input)
    root = tree.getroot()

    # Extract namespace automatically
    ns = {'ns': root.tag.split('}')[0].strip('{')}

    # Locate the <SurveyPlan> container
    survey_plan = root.find('.//ns:SurveyPlan', ns)
    if survey_plan is None:
        raise RuntimeError("Could not find <SurveyPlan> in XML.")

    # Extract all SurveyPlanStep elements
    steps = survey_plan.findall('ns:SurveyPlanStep', ns)

    # Sort by numeric PassPlan value
    def get_passplan_number(step):
        pp = step.find('ns:PassPlan', ns)
        return int(pp.text.strip()) if pp is not None else 999999999

    steps_sorted = sorted(steps, key=get_passplan_number)

    # Clear existing order
    for step in steps:
        survey_plan.remove(step)

    # Reinsert in sorted order
    for step in steps_sorted:
        survey_plan.append(step)

    # Save output
    tree.write(xml_output, encoding="UTF-8", xml_declaration=True)
    print(f"SurveyPlan sorted and saved to {xml_output}")



def update_orient_ranges(xml_input, xml_output, orient_min_list, orient_max_list):
    """
    Updates OrientRange OrientMin/OrientMax in each SurveyPlanStep using
    user-provided arrays.
    
    orient_min_list and orient_max_list must have the same length as the
    number of SurveyPlanStep entries.
    """

    tree = ET.parse(xml_input)
    root = tree.getroot()

    # Extract namespace automatically
    ns = {'ns': root.tag.split('}')[0].strip('{')}

    # Find all SurveyPlanStep entries
    steps = root.findall('.//ns:SurveyPlan/ns:SurveyPlanStep', ns)

    if len(steps) != len(orient_min_list) or len(steps) != len(orient_max_list):
        raise ValueError("ERROR: Input arrays must match number of SurveyPlanStep entries.")

    # Update OrientMin and OrientMax for each step
    for step, new_min, new_max in zip(steps, orient_min_list, orient_max_list):
        orient_range = step.find('ns:SpecialRequirements/ns:OrientRange', ns)
        if orient_range is not None:
            orient_range.set("OrientMin", f"{new_min} Degrees")
            orient_range.set("OrientMax", f"{new_max} Degrees")
        else:
            print(f"Warning: No OrientRange found in SurveyPlanStep uid={step.get('uid')}")

    # Save updated XML
    tree.write(xml_output, encoding="UTF-8", xml_declaration=True)
    print(f"Updated XML saved to {xml_output}")


# Example usage:
# new_min = [142.1, 143.2, 144.3, ...]
# new_max = [142.1, 143.2, 144.3, ...]


In [ ]:
sync_passplan_numbers(xml_input="CAR171.apt", xml_output="CAR171_sync.apt")
sort_surveyplan_steps(xml_input="CAR171_sync.apt", xml_output="CAR171_order.apt")
update_target_coordinates(xml_path="CAR171_order.apt", output_path="CAR171_mod.apt", new_coords=coords)
update_orient_ranges(xml_input="CAR171_mod.apt", xml_output="CAR171_orient.apt", orient_min_list=CAR_dat["V3PA_off"], orient_max_list=CAR_dat["V3PA_off"])

In [ ]:
coords